In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

# Workshop: Training Faster R-CNN on a Custom Dataset with PyTorch

In this workshop we will train a **Faster R-CNN** object detector from scratch (well, from pre-trained ImageNet weights) on a small real-world dataset.

### What we'll cover

1. **Dataset** – Download and explore the [Penn-Fudan Pedestrian Detection](https://www.cis.upenn.edu/~jshi/ped_html/) dataset (~170 images, 1 class: `pedestrian`).
2. **Custom Dataset class** – How to write a `torch.utils.data.Dataset` that produces bounding-box targets in the format Faster R-CNN expects.
3. **Model setup** – Load a pre-trained `fasterrcnn_resnet50_fpn_v2` from `torchvision` and replace its classification head for our number of classes.
4. **Training loop** – A clean training loop that logs losses and saves a checkpoint.
5. **Evaluation & visualisation** – Run inference on held-out images and draw predicted bounding boxes.

### Why Faster R-CNN?

| Property | Faster R-CNN |
|---|---|
| Architecture family | Two-stage detector |
| Key component | Region Proposal Network (RPN) |
| Speed | ~5 fps (GPU) – not real-time, but very accurate |
| Torchvision support | ✅ built-in, easy to fine-tune |

Faster R-CNN remains the go-to baseline for many academic benchmarks and industrial applications where accuracy matters more than raw inference speed.

## 1. Installation & Imports

In [ ]:
# Uncomment if you need to install dependencies
# !pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install -U matplotlib pillow tqdm

In [ ]:
import os
from pathlib import Path
import zipfile
import urllib.request
from typing import Callable, Optional

import numpy as np
import torch
import torchvision
import torchvision.transforms.v2 as T
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

print(f"torch      : {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")

## 2. Device Setup

We auto-detect the best available device: CUDA GPU → Apple MPS → CPU.

In [ ]:
def get_device() -> torch.device:
    return torch.device(
        'cuda' if torch.cuda.is_available() else (
            'mps' if torch.backends.mps.is_available() else 'cpu'
        )
    )


DEVICE = get_device()
print(f"Using device: {DEVICE}")

## 3. Dataset: Penn-Fudan Pedestrian

The **Penn-Fudan Pedestrian** dataset contains 170 images (train + val) with pixel-level segmentation masks for every pedestrian. We'll use the masks to derive tight **bounding boxes** for our object detection task.

- 170 images total → we'll use 150 for training and 20 for validation
- 1 foreground class: `pedestrian` (class id = 1; class id 0 is always background in Faster R-CNN)
- Freely downloadable from UPenn

### Dataset download

In [ ]:
DATA = Path("data") 
DATASET = DATA / "dataset"
DATA_ROOT = DATASET / "PennFudanPed"
ZIP_PATH  = DATASET / "PennFudanPed.zip"
ZIP_URL   = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATA_ROOT.exists() and any(DATA_ROOT.iterdir()):
    print(f"Dataset already exists at {DATA_ROOT}")
else:
    ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading Penn-Fudan Pedestrian dataset...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("datasets/")
    print(f"Dataset ready at {DATA_ROOT}")
    

# Check what's inside
for sub in sorted(DATA_ROOT.iterdir()):
    n = len(list(sub.iterdir()))
    print(f"  {sub.name}/  ({n} files)")

### Explore a sample image & mask

Each image in `PNGImages/` is paired with a segmentation mask in `PedMasks/`. The mask stores each pedestrian instance with a unique integer label (1, 2, 3, …). We'll use `np.unique` to extract per-instance bounding boxes.

In [ ]:
img_dir  = DATA_ROOT / "PNGImages"
mask_dir = DATA_ROOT / "PedMasks"

# Pick the first sample
sample_img_path  = sorted(img_dir.glob("*.png"))[0]
sample_mask_path = mask_dir / (sample_img_path.stem + "_mask.png")

img  = np.array(Image.open(sample_img_path))
mask = np.array(Image.open(sample_mask_path))

print(f"Image shape : {img.shape}")
print(f"Mask shape  : {mask.shape}")
print(f"Mask values : {np.unique(mask)}  (0 = background, 1+ = pedestrian instances)")

# Derive bounding boxes from the mask
obj_ids  = np.unique(mask)[1:]  # skip 0 (background)
masks_3d = mask == obj_ids[:, None, None]  # (N, H, W) binary masks

boxes = []
for m in masks_3d:
    pos = np.where(m)
    xmin, xmax = pos[1].min(), pos[1].max()
    ymin, ymax = pos[0].min(), pos[0].max()
    boxes.append([xmin, ymin, xmax, ymax])

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(img);          axes[0].set_title("Image");        axes[0].axis("off")
axes[1].imshow(mask, cmap="tab10"); axes[1].set_title("Mask"); axes[1].axis("off")

axes[2].imshow(img)
for xmin, ymin, xmax, ymax in boxes:
    rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                               linewidth=2, edgecolor="lime", facecolor="none")
    axes[2].add_patch(rect)
axes[2].set_title(f"Bounding boxes ({len(boxes)} pedestrians)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
img.shape

In [ ]:
torchvision.io.read_image(sample_img_path).shape

## 4. Custom `Dataset` Class

Faster R-CNN in `torchvision` expects each sample's target to be a **dict** with at minimum:

```python
{
    "boxes"  : FloatTensor[N, 4],   # [x_min, y_min, x_max, y_max] in pixel coords
    "labels" : Int64Tensor[N],      # class ids (0 = background is excluded here)
}
```

Optional but useful fields: `image_id`, `area`, `iscrowd`.

In [ ]:
class PennFudanDataset(torch.utils.data.Dataset):
    """Penn-Fudan Pedestrian dataset for Faster R-CNN."""

    def __init__(
        self,
        root: Path,
        transforms: Optional[Callable] = None,
    ):
        self.root       = Path(root)
        self.transforms = transforms
        self.img_paths  = sorted((self.root / "PNGImages").glob("*.png"))
        self.mask_dir   = self.root / "PedMasks"

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int):
        img_path  = self.img_paths[idx]
        mask_path = self.mask_dir / (img_path.stem + "_mask.png")

        # Load image as RGB tensor
        image = torchvision.io.read_image(str(img_path))  # (C, H, W) uint8
        image = image[:3]  # ensure 3 channels (drop alpha if present)

        # Load mask
        mask = np.array(Image.open(mask_path))
        obj_ids = np.unique(mask)[1:]  # skip background (0)

        # Derive bounding boxes
        masks_3d = mask == obj_ids[:, None, None]  # (N, H, W)
        boxes, valid = [], []
        for i, m in enumerate(masks_3d):
            pos  = np.where(m)
            xmin, xmax = int(pos[1].min()), int(pos[1].max())
            ymin, ymax = int(pos[0].min()), int(pos[0].max())
            # Skip degenerate boxes (width or height == 0)
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                valid.append(i)

        boxes  = torch.as_tensor(boxes, dtype=torch.float32)  # (N, 4)
        labels = torch.ones(len(boxes), dtype=torch.int64)     # class 1 = pedestrian
        area   = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) else torch.zeros(0)

        target = {
            "boxes"    : boxes,
            "labels"   : labels,
            "image_id" : torch.tensor([idx]),
            "area"     : area,
            "iscrowd"  : torch.zeros(len(boxes), dtype=torch.int64),
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target

## 5. Data Transforms

We use `torchvision.transforms.v2` (the new API) which is **bounding-box-aware** — it automatically adjusts box coordinates when flipping or cropping the image, so we don't have to do this manually.

- **Train**: random horizontal flip + convert to float
- **Val**: convert to float only

In [ ]:
def get_transform(train: bool):
    ops = [
        T.ToDtype(torch.float32, scale=True),  # uint8 [0,255] → float32 [0,1]
    ]
    if train:
        ops.insert(0, T.RandomHorizontalFlip(p=0.5))
    return T.Compose(ops)


# Quick sanity check
ds_check = PennFudanDataset(DATA_ROOT, transforms=get_transform(train=True))
img_t, tgt_t = ds_check[0]
print(f"image dtype : {img_t.dtype}, shape: {img_t.shape}")
print(f"boxes shape : {tgt_t['boxes'].shape}")
print(f"labels      : {tgt_t['labels']}")

## 6. DataLoaders

Faster R-CNN accepts batches where each element can have a **different number of boxes**, so we use a custom `collate_fn` that keeps the targets as a list instead of stacking them into a single tensor.

In [ ]:
def collate_fn(batch):
    """Keep images and targets as separate tuples (no stacking of targets)."""
    return tuple(zip(*batch))


TRAIN_SPLIT = 150
BATCH_SIZE  = 2

dataset_full = PennFudanDataset(DATA_ROOT)
indices      = torch.randperm(len(dataset_full)).tolist()

dataset_train = torch.utils.data.Subset(
    PennFudanDataset(DATA_ROOT, transforms=get_transform(train=True)),
    indices[:TRAIN_SPLIT],
)
dataset_val = torch.utils.data.Subset(
    PennFudanDataset(DATA_ROOT, transforms=get_transform(train=False)),
    indices[TRAIN_SPLIT:],
)

train_loader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)
val_loader = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print(f"Train samples : {len(dataset_train)}")
print(f"Val   samples : {len(dataset_val)}")
print(f"Train batches : {len(train_loader)}")

In [ ]:
dataset_full[0]

## 7. Model: Faster R-CNN with ResNet-50 FPN

We use `torchvision.models.detection.fasterrcnn_resnet50_fpn_v2` with COCO-pretrained weights.

The model has two heads:
- **RPN head** – predicts object/background for anchor boxes
- **RoI head** – predicts class and bounding box offsets for each proposed region

We only replace the **box predictor** (the final linear layers of the RoI head) so that the number of output classes matches our dataset. The RPN stays unchanged since it's class-agnostic.

```
num_classes = 2   ← background (0) + pedestrian (1)
```

In [ ]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def build_faster_rcnn(num_classes: int) -> torch.nn.Module:
    """Load COCO-pretrained Faster R-CNN and replace the head for `num_classes`."""
    # Load with COCO weights (80 COCO classes + background = 91 outputs)
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )

    # Get the input feature size of the existing classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Replace with a fresh predictor for our number of classes
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model


NUM_CLASSES = 2  # background + pedestrian
model = build_faster_rcnn(NUM_CLASSES)
model.to(DEVICE)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

## 8. Optimizer & Learning-Rate Scheduler

We use **SGD with momentum** (the classic Faster R-CNN training recipe).

- Initial LR: `0.005`
- Momentum: `0.9`, weight decay: `5e-4`
- **StepLR** scheduler: drops LR by 10× every 3 epochs

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=5e-4,
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1,
)

print("Optimizer ready:", optimizer)

## 9. Training Loop

### How training works in torchvision's Faster R-CNN

When the model is in **train mode** and you pass both images *and* targets, it returns a **dict of losses** (no predictions). These losses are:

| Key | Meaning |
|---|---|
| `loss_objectness` | RPN binary classification (object vs background) |
| `loss_rpn_box_reg` | RPN box regression |
| `loss_classifier` | RoI classification |
| `loss_box_reg` | RoI box regression |

We sum all four and backpropagate.

In [ ]:
def train_one_epoch(model, optimizer, loader, device, epoch):
    """Run one training epoch and return the mean total loss."""
    model.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch:02d} [train]", leave=False)
    for images, targets in pbar:
        # Move to device
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass → dict of losses
        loss_dict = model(images, targets)
        losses    = sum(loss_dict.values())

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
        pbar.set_postfix(loss=f"{losses.item():.4f}")

    return total_loss / len(loader)

In [ ]:
NUM_EPOCHS = 5
CHECKPOINT_DIR = Path("models/faster_rcnn_pennfudan")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

history = []

for epoch in range(1, NUM_EPOCHS + 1):
    mean_loss = train_one_epoch(model, optimizer, train_loader, DEVICE, epoch)
    lr_scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history.append({"epoch": epoch, "loss": mean_loss, "lr": current_lr})
    print(f"Epoch {epoch:02d}  |  loss: {mean_loss:.4f}  |  lr: {current_lr:.6f}")

    # Save checkpoint after every epoch
    ckpt_path = CHECKPOINT_DIR / f"epoch_{epoch:02d}.pth"
    torch.save(
        {
            "epoch"       : epoch,
            "model_state" : model.state_dict(),
            "optim_state" : optimizer.state_dict(),
            "loss"        : mean_loss,
        },
        ckpt_path,
    )

print("\nTraining complete! ✓")

### Training loss curve

In [ ]:
epochs = [h["epoch"] for h in history]
losses = [h["loss"]  for h in history]

plt.figure(figsize=(8, 4))
plt.plot(epochs, losses, marker="o", linewidth=2, color="steelblue")
plt.xlabel("Epoch")
plt.ylabel("Mean Total Loss")
plt.title("Faster R-CNN Training Loss (Penn-Fudan)")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 10. Inference & Visualisation

Switch the model to `eval()` mode. In eval mode, the model returns a list of dicts (one per image), each containing:

```python
{
    "boxes"  : FloatTensor[K, 4],
    "labels" : Int64Tensor[K],
    "scores" : FloatTensor[K],
}
```

We filter predictions with score > threshold before drawing.

In [ ]:
CLASS_NAMES = ["__background__", "pedestrian"]
SCORE_THRESHOLD = 0.5


def draw_boxes(ax, boxes, labels, scores, color="lime"):
    for box, label, score in zip(boxes, labels, scores):
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor=color, facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            xmin, ymin - 4,
            f"{CLASS_NAMES[label]}: {score:.2f}",
            color="white", fontsize=8,
            bbox=dict(facecolor=color, alpha=0.6, pad=1, edgecolor="none"),
        )


@torch.inference_mode()
def predict(model, image_tensor, device, threshold=0.5):
    model.eval()
    img = image_tensor.to(device).float() / 255.0  # uint8 → float
    outputs = model([img])[0]
    # Filter by score threshold
    keep = outputs["scores"] >= threshold
    return {
        "boxes" : outputs["boxes"][keep].cpu().numpy(),
        "labels": outputs["labels"][keep].cpu().numpy(),
        "scores": outputs["scores"][keep].cpu().numpy(),
    }

In [ ]:
# Use a raw (untransformed) val dataset so we get the original uint8 image
dataset_val_raw = torch.utils.data.Subset(
    PennFudanDataset(DATA_ROOT, transforms=None),
    indices[TRAIN_SPLIT:],
)

N_SHOW = min(6, len(dataset_val_raw))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i in range(N_SHOW):
    raw_img, gt_target = dataset_val_raw[i]   # uint8 tensor

    preds = predict(model, raw_img, DEVICE, threshold=SCORE_THRESHOLD)

    # Convert (C, H, W) uint8 → (H, W, C) for matplotlib
    display_img = raw_img.permute(1, 2, 0).numpy()

    axes[i].imshow(display_img)

    # Draw ground truth (blue)
    gt_boxes  = gt_target["boxes"].numpy()
    gt_labels = gt_target["labels"].numpy()
    gt_scores = np.ones(len(gt_boxes))
    draw_boxes(axes[i], gt_boxes,  gt_labels,  gt_scores,  color="deepskyblue")

    # Draw predictions (lime green)
    draw_boxes(axes[i], preds["boxes"], preds["labels"], preds["scores"], color="lime")

    n_pred = len(preds["boxes"])
    n_gt   = len(gt_boxes)
    axes[i].set_title(f"Sample {i+1} | GT: {n_gt} (blue) | Pred: {n_pred} (green)")
    axes[i].axis("off")

plt.suptitle(
    f"Faster R-CNN predictions after {NUM_EPOCHS} epochs (score ≥ {SCORE_THRESHOLD})",
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.show()

## 11. Quick mAP Evaluation

We compute **mAP@0.50** (PASCAL VOC-style) using `torchmetrics.detection.MeanAveragePrecision`. This metric tells us how well the predicted boxes overlap with ground-truth boxes (IoU ≥ 0.50).

In [ ]:
# Install torchmetrics if needed
# !pip install -q torchmetrics

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    HAS_TORCHMETRICS = True
except ImportError:
    print("torchmetrics not installed – skipping mAP computation.")
    print("Install with: pip install torchmetrics")
    HAS_TORCHMETRICS = False

In [ ]:
if HAS_TORCHMETRICS:
    metric = MeanAveragePrecision(iou_type="bbox")
    model.eval()

    with torch.inference_mode():
        for images, targets in tqdm(val_loader, desc="Computing mAP"):
            images  = [img.to(DEVICE) for img in images]
            outputs = model(images)

            # torchmetrics expects lists of dicts on CPU
            preds_tm = [
                {
                    "boxes" : o["boxes"].cpu(),
                    "scores": o["scores"].cpu(),
                    "labels": o["labels"].cpu(),
                }
                for o in outputs
            ]
            targets_tm = [
                {
                    "boxes" : t["boxes"].cpu(),
                    "labels": t["labels"].cpu(),
                }
                for t in targets
            ]
            metric.update(preds_tm, targets_tm)

    results = metric.compute()
    print(f"\nmAP@0.50:0.95 : {results['map']:.4f}")
    print(f"mAP@0.50      : {results['map_50']:.4f}")
    print(f"mAP@0.75      : {results['map_75']:.4f}")

## 12. Load a Saved Checkpoint

This is how you reload a checkpoint after restarting the kernel.

In [ ]:
# Example: reload the final epoch checkpoint
# ckpt = torch.load(CHECKPOINT_DIR / f"epoch_{NUM_EPOCHS:02d}.pth", map_location=DEVICE)
# model.load_state_dict(ckpt["model_state"])
# print(f"Loaded checkpoint from epoch {ckpt['epoch']}, loss={ckpt['loss']:.4f}")

## 13. Exercises

Try these to deepen your understanding:

1. **More epochs** – Increase `NUM_EPOCHS` to 10-15. Does mAP improve?
2. **Data augmentation** – Add `T.ColorJitter`, `T.RandomPhotometricDistort`, or `T.RandomZoomOut` to the training transforms and observe the effect on overfitting.
3. **Freeze the backbone** – Set `requires_grad=False` for the `model.backbone` parameters and train only the heads. How does this affect training speed and final accuracy?
4. **Different backbone** – Replace `fasterrcnn_resnet50_fpn_v2` with `fasterrcnn_mobilenet_v3_large_fpn` for a faster/lighter model.
5. **Score threshold** – Experiment with different `SCORE_THRESHOLD` values (0.3, 0.7) and observe precision vs. recall trade-off.
6. **Your own dataset** – Adapt `PennFudanDataset` to a different dataset (COCO subset, VOC, or your own annotated data).

## Conclusion

In this workshop we:

- Wrote a custom `Dataset` that derives bounding boxes from segmentation masks
- Used `torchvision.transforms.v2` for box-aware data augmentation
- Fine-tuned a COCO-pretrained **Faster R-CNN ResNet-50 FPN v2** on a small pedestrian detection dataset
- Visualised predictions vs. ground truth
- Computed mAP with `torchmetrics`

Even with just 5 epochs and 150 training images, transfer learning from COCO gives us a solid starting point. Happy detecting!